In [2]:
!pip install torch transformers pytesseract opencv-python rouge-score pillow numpy==1.26.4 pandas

In [ ]:
import re
import os
import cv2
import pytesseract
import numpy as np
import pandas as pd

from transformers import pipeline
from rouge_score import rouge_scorer

print("✅ Setup Complete")

✅ Setup Complete


Text Cleaning


In [4]:
def clean_text(text):
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

OCR MODULE (IMAGE → TEXT)

In [5]:
import cv2
import pytesseract
import os

def extract_text_with_debug(image_path):
    if not os.path.exists(image_path):
        raise FileNotFoundError("Image not found")

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Invalid image")

    raw_text = pytesseract.image_to_string(img)

    img = cv2.resize(img, None, fx=2, fy=2)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)

    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

    processed_text = pytesseract.image_to_string(
        thresh, config="--oem 3 --psm 6"
    )

    return raw_text, processed_text

LSTM BASELINE (FOR COMPARISON)


In [6]:
def lstm_summary(text):
    sentences = text.split(".")
    return ". ".join(sentences[:2])

In [7]:
print("⏳ Loading models...")

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

print("✅ Models Loaded")

⏳ Loading models...



Device set to use cpu
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


✅ Models Loaded


TRANSFORMER SUMMARIZATION

In [8]:
def generate_summary(text):
    if not text:
        return ""

    max_chunk = 500
    chunks = [text[i:i+max_chunk] for i in range(0, len(text), max_chunk)]

    summaries = []

    for chunk in chunks[:3]:
        if len(chunk.strip()) < 50:
            continue

        out = summarizer(chunk, max_length=100, min_length=30)
        summaries.append(out[0]['summary_text'])

    return " ".join(summaries)

NER

In [9]:
def extract_entities(text):
    results = ner(text)

    return [
        {
            "text": r["word"],
            "label": r["entity_group"],
            "confidence": round(r["score"], 3)
        }
        for r in results
    ]

In [10]:
def diabetes_entities(text):
    text = text.lower()
    entities = []

    if "diabetes" in text:
        entities.append("Disease: Diabetes")
    if "insulin" in text:
        entities.append("Drug: Insulin")
    if "blood sugar" in text:
        entities.append("Symptom: High Blood Sugar")

    return entities

In [11]:
def run_pipeline_text(text):
    cleaned = clean_text(text)

    summary = generate_summary(cleaned)
    entities = extract_entities(summary)
    medical = diabetes_entities(summary)

    return pd.DataFrame({
        "Raw Text": [text],
        "Summary": [summary],
        "Entities": [entities],
        "Medical": [medical]
    })

In [12]:
def run_pipeline_image(image_path):
    raw, processed = extract_text_with_debug(image_path)

    print("\n🖼️ BEFORE OCR:\n", raw)
    print("\n🧹 AFTER OCR:\n", processed)

    return run_pipeline_text(processed)

ROUGE EVALUATION

In [13]:
def compute_rouge(reference, generated):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(reference, generated)

    return {
        "R1": scores['rouge1'].fmeasure,
        "R2": scores['rouge2'].fmeasure,
        "RL": scores['rougeL'].fmeasure
    }


def compare_models(text, reference):
    lstm_out = lstm_summary(text)
    bart_out = generate_summary(text)

    lstm_scores = compute_rouge(reference, lstm_out)
    bart_scores = compute_rouge(reference, bart_out)

    print("\n📊 ROUGE COMPARISON\n")
    print(f"{'Model':<15}{'R1':<10}{'R2':<10}{'RL':<10}")
    print("-"*40)

    print(f"{'LSTM':<15}{lstm_scores['R1']:.3f}"
          f"{lstm_scores['R2']:.3f}{lstm_scores['RL']:.3f}")

    print(f"{'BART':<15}{bart_scores['R1']:.3f}"
          f"{bart_scores['R2']:.3f}{bart_scores['RL']:.3f}")

In [14]:
text = """
Patient is a 50 year old male from Chennai suffering from diabetes.
Blood sugar levels are high. Prescribed insulin and metformin.
"""

run_pipeline_text(text)

Your max_length is set to 100, but your input_length is only 27. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)


,Raw Text,Summary,Entities,Medical
0,\nPatient is a 50 year old male from Chennai s...,The patient is a 50 year old male from chennai...,"[{'text': 'ch', 'label': 'LOC', 'confidence': ...","[Disease: Diabetes, Drug: Insulin, Symptom: Hi..."


In [ ]:
run_pipeline_image("sample_prescription.jpg")

In [15]:
text = "Patient has diabetes. Blood sugar high. Insulin given."
reference = "Patient diagnosed with diabetes and treated with insulin."

compare_models(text, reference)

Your max_length is set to 100, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



📊 ROUGE COMPARISON

Model          R1        R2        RL        
----------------------------------------
LSTM           0.2860.0000.286
BART           0.3030.0000.242


Without Attention: Entire text compressed → loss of info

With Attention: Focus on important words → better summary

